# Example: Symmetry-reduced Pulse Design Tool

This notebook is based on example 11, but prepares the Pulse Design Tool on an
exactly up-down symmetric machine and plasma. It then removes every odd-in-Z
passive mode before calculating plasma coupling.

Exact symmetry is imposed on:

- the active circuits, with the odd `P6` circuit omitted;
- all passive structures and their quadrature filaments;
- the limiter and wall;
- the initial active and passive currents;
- every static Grad-Shafranov solve during the evolution.

This is a deliberately strict first implementation. It models only symmetric
evolution; prescribed asymmetric drives and odd passive-current responses are
outside its scope.

### Generate an exactly symmetric starting equilibrium

The first eleven MAST-U-like active circuits are even in Z. The odd `P6` circuit
is removed from this machine description, rather than retained as a permanently
zero dynamical variable. The passive polygons and limiter are close to symmetric,
but not exact, so this example mirrors one half of each description explicitly.

The passive polygon refinement currently samples each polygon independently.
After building the machine, the quadrature filaments of each lower passive
structure are therefore mirrored exactly into its upper partner and the
resistance and inductance matrices are rebuilt.

In [1]:
# packages
from copy import deepcopy
import pickle
import time

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolygonPatch
import numpy as np
from scipy.optimize import linear_sum_assignment
from shapely.affinity import scale
from shapely.geometry import Polygon, box
from shapely.ops import unary_union

from freegsnke import build_machine, equilibrium_update, GSstaticsolver
from freegsnke.machine_config import build_tokamak_R_and_M


#### Symmetric machine-description helpers

The helpers pair the existing lower and upper passive polygons geometrically,
use each lower polygon as the source for an exact upper mirror, and construct
the limiter and wall from an upper half and its reflection.

`mirror_passive_quadrature` is needed because independently generated polygon
quadrature points need not be exact mirrors, even when the polygon vertices are.
It does not change the number of passive structures or their current variables.


In [2]:
def structure_polygon(structure):
    """Return a Shapely polygon for one passive-structure description."""
    return Polygon(np.column_stack((structure["R"], structure["Z"])))


def make_symmetric_passives(passive_data):
    """Return exact lower/upper passive mirrors and their index pairs."""
    lower_indices = [
        i
        for i, structure in enumerate(passive_data)
        if structure_polygon(structure).centroid.y < 0
    ]
    upper_indices = [
        i
        for i, structure in enumerate(passive_data)
        if structure_polygon(structure).centroid.y > 0
    ]

    if len(lower_indices) != len(upper_indices):
        raise ValueError(
            "The source description must contain equal numbers of lower and upper "
            "passive structures, with no structure crossing Z=0."
        )

    pairing_cost = np.empty((len(lower_indices), len(upper_indices)))
    for lower_position, lower_index in enumerate(lower_indices):
        reflected_lower = scale(
            structure_polygon(passive_data[lower_index]),
            xfact=1,
            yfact=-1,
            origin=(0, 0),
        )
        for upper_position, upper_index in enumerate(upper_indices):
            pairing_cost[lower_position, upper_position] = (
                reflected_lower.hausdorff_distance(
                    structure_polygon(passive_data[upper_index])
                )
            )

    lower_positions, upper_positions = linear_sum_assignment(pairing_cost)
    symmetric_data = deepcopy(passive_data)
    passive_pairs = []

    for lower_position, upper_position in zip(lower_positions, upper_positions):
        lower_index = lower_indices[lower_position]
        upper_index = upper_indices[upper_position]
        source = passive_data[lower_index]

        lower = deepcopy(source)
        upper = deepcopy(source)

        # Preserve labels and grouping metadata while copying all physical data
        # from one side of the machine.
        for key in ("name", "efitGroup", "element"):
            if key in passive_data[lower_index]:
                lower[key] = passive_data[lower_index][key]
            if key in passive_data[upper_index]:
                upper[key] = passive_data[upper_index][key]

        # Reverse vertex order so reflection preserves polygon winding.
        upper["R"] = list(np.asarray(source["R"], dtype=float)[::-1])
        upper["Z"] = list(-np.asarray(source["Z"], dtype=float)[::-1])

        symmetric_data[lower_index] = lower
        symmetric_data[upper_index] = upper
        passive_pairs.append((lower_index, upper_index))

    return symmetric_data, passive_pairs


def make_symmetric_boundary(boundary_data):
    """Build an exact symmetric closed boundary from its original upper half."""
    polygon = Polygon([(point["R"], point["Z"]) for point in boundary_data])
    min_r, _, max_r, max_z = polygon.bounds
    upper = polygon.intersection(box(min_r - 1, 0, max_r + 1, max_z + 1))
    reflected_upper = scale(upper, xfact=1, yfact=-1, origin=(0, 0))
    symmetric = unary_union((upper, reflected_upper))

    if symmetric.geom_type != "Polygon":
        raise ValueError("The symmetrised boundary is not a single closed polygon.")

    return [
        {"R": float(r_value), "Z": float(z_value)}
        for r_value, z_value in list(symmetric.exterior.coords)[:-1]
    ]


def mirror_passive_quadrature(tokamak, passive_data, passive_pairs):
    """Mirror lower passive quadrature filaments and rebuild machine matrices."""
    for lower_index, upper_index in passive_pairs:
        lower_name = passive_data[lower_index]["name"]
        upper_name = passive_data[upper_index]["name"]
        lower = tokamak[lower_name]
        upper = tokamak[upper_name]

        upper._area = lower._area
        upper.R = lower.R
        upper.Z = -lower.Z
        upper.Len = lower.Len
        upper.n_refine = lower.n_refine
        upper.filaments = np.column_stack(
            (lower.filaments[:, 0], -lower.filaments[:, 1])
        )
        upper.greens = {}

        upper.Rpolygon = np.asarray(passive_data[upper_index]["R"])
        upper.Zpolygon = np.asarray(passive_data[upper_index]["Z"])
        upper.vertices = np.column_stack((upper.Rpolygon, upper.Zpolygon))
        upper.polygon = PolygonPatch(upper.vertices, facecolor="k", alpha=0.75)

        lower_data = tokamak.coils_dict[lower_name]
        upper_data = tokamak.coils_dict[upper_name]
        upper_data["vertices"] = np.array((upper.Rpolygon, upper.Zpolygon))
        upper_data["coords"] = np.array(
            (upper.filaments[:, 0], upper.filaments[:, 1])
        )
        for key in (
            "area",
            "dR",
            "dZ",
            "polarity",
            "multiplier",
            "resistivity_over_area",
        ):
            upper_data[key] = deepcopy(lower_data[key])

    # These matrices were built before replacing the upper quadrature points.
    del tokamak.coil_resist
    del tokamak.coil_self_ind
    build_tokamak_R_and_M(tokamak)


def relative_reflection_error(values, parity=1):
    """Return ||f - parity*reflect(f)|| / ||f|| for a 2D field."""
    denominator = np.linalg.norm(values)
    if denominator == 0:
        return 0.0
    return np.linalg.norm(values - parity * values[:, ::-1]) / denominator


In [3]:
# Load the familiar example-11 descriptions as source data.
machine_path = "../machine_configs/MAST-U"
with open(f"{machine_path}/MAST-U_like_active_coils.pickle", "rb") as handle:
    active_coils_data = pickle.load(handle)

# P6 is odd in Z, so it is outside the strictly symmetric dynamical subspace.
active_coils_data = deepcopy(active_coils_data)
active_coils_data.pop("P6")
with open(f"{machine_path}/MAST-U_like_passive_coils.pickle", "rb") as handle:
    passive_coils_source = pickle.load(handle)
with open(f"{machine_path}/MAST-U_like_limiter.pickle", "rb") as handle:
    limiter_source = pickle.load(handle)
with open(f"{machine_path}/MAST-U_like_wall.pickle", "rb") as handle:
    wall_source = pickle.load(handle)

symmetric_passives, passive_pairs = make_symmetric_passives(passive_coils_source)
symmetric_limiter = make_symmetric_boundary(limiter_source)
symmetric_wall = make_symmetric_boundary(wall_source)

tokamak = build_machine.tokamak(
    active_coils_data=active_coils_data,
    passive_coils_data=symmetric_passives,
    limiter_data=symmetric_limiter,
    wall_data=symmetric_wall,
)
mirror_passive_quadrature(tokamak, symmetric_passives, passive_pairs)

# Use a grid whose Z coordinates are exact reflection pairs about Z=0.
eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1,
    Rmax=2.0,
    Zmin=-2.2,
    Zmax=2.2,
    nx=65,
    ny=65,
)

from freegsnke.jtor_update import Lao85

profiles = Lao85(
    eq=eq,
    Ip=7.59e5,
    fvac=-0.52,
    alpha=[362685, 17696],
    beta=[0.103, 0.753],
    alpha_logic=True,
    beta_logic=True,
)

# All retained active circuits are even. Passive currents are zero by
# construction.
coil_currents = np.array(
    [
        2766.10782056,
        193.12768191,
        4019.86873871,
        4857.42357466,
        -726.99282376,
        -1696.92521166,
        -77.7252396,
        266.38207493,
        -73.01483716,
        -4169.86874271,
        -4518.28147417,
    ]
)
for coil_name, current in zip(tokamak.coil_names[:11], coil_currents):
    eq.tokamak.set_coil_current(coil_label=coil_name, current_value=current)

GSStaticSolver = GSstaticsolver.NKGSsolver(eq)
GSStaticSolver.solve(
    eq=eq,
    profiles=profiles,
    constrain=None,
    target_relative_tolerance=1e-8,
    force_up_down_symmetric=True,
)


Active coils --> built from user-provided data.
Passive structures --> built from user-provided data.
Limiter --> built from user-provided data.
Wall --> built from user-provided data.


Magnetic probes --> none provided.


Resistance (R) and inductance (M) matrices --> built using actives (and passives if present).
Tokamak built.


Resistance (R) and inductance (M) matrices --> built using actives (and passives if present).


Forward static solve SUCCESS. Tolerance 6.31e-09 (vs. requested 1.00e-08) reached in 23/100 iterations.


#### Validate the exact symmetry

The full reflection operator leaves each active circuit unchanged and swaps
every lower/upper passive pair. Its passive block is also passed to the
evolutive solver, which uses it to classify passive normal modes and discard
the odd modes before calculating plasma coupling.

The machine operators, vacuum field, limiter mask, plasma fields, and initial
current state are checked independently. Errors should be at floating-point
round-off.

In [4]:
n_coils = tokamak.n_coils
n_active = tokamak.n_active_coils
n_passive = n_coils - n_active

passive_reflection_operator = np.zeros((n_passive, n_passive))
for lower_index, upper_index in passive_pairs:
    passive_reflection_operator[lower_index, upper_index] = 1
    passive_reflection_operator[upper_index, lower_index] = 1

reflection_operator = np.zeros((n_coils, n_coils))
reflection_operator[:n_active, :n_active] = np.eye(n_active)
reflection_operator[n_active:, n_active:] = passive_reflection_operator

active_green_errors = [
    relative_reflection_error(eq._vgreen[i]) for i in range(n_active)
]
passive_green_errors = [
    np.linalg.norm(
        eq._vgreen[n_active + lower_index]
        - eq._vgreen[n_active + upper_index][:, ::-1]
    )
    / np.linalg.norm(eq._vgreen[n_active + lower_index])
    for lower_index, upper_index in passive_pairs
]

resistance_matrix = np.diag(tokamak.coil_resist)
machine_timescale_matrix = np.linalg.solve(
    resistance_matrix, tokamak.coil_self_ind
)


def relative_commutator_error(matrix):
    return (
        np.linalg.norm(
            reflection_operator @ matrix - matrix @ reflection_operator
        )
        / np.linalg.norm(matrix)
    )


initial_currents = tokamak.getCurrentsVec()
symmetry_errors = {
    "active Green functions": max(active_green_errors),
    "passive Green functions": max(passive_green_errors),
    "inductance operator": relative_commutator_error(tokamak.coil_self_ind),
    "resistance operator": relative_commutator_error(resistance_matrix),
    "machine timescale operator": relative_commutator_error(
        machine_timescale_matrix
    ),
    "vacuum flux": relative_reflection_error(
        tokamak.getPsitokamak(vgreen=eq._vgreen)
    ),
    "plasma flux": relative_reflection_error(eq.plasma_psi),
    "toroidal current density": relative_reflection_error(profiles.jtor),
    "initial metal currents": (
        np.linalg.norm(reflection_operator @ initial_currents - initial_currents)
        / np.linalg.norm(initial_currents)
    ),
}
limiter_mismatch = np.count_nonzero(
    eq.limiter_handler.mask_inside_limiter
    != eq.limiter_handler.mask_inside_limiter[:, ::-1]
)

for name, error in symmetry_errors.items():
    print(f"{name:28s}: {error:.3e}")
print(f"{'limiter-mask mismatches':28s}: {limiter_mismatch:d}")

symmetry_tolerance = 1e-12
assert max(symmetry_errors.values()) < symmetry_tolerance
assert limiter_mismatch == 0

active Green functions      : 2.940e-15
passive Green functions     : 2.960e-15
inductance operator         : 1.070e-17
resistance operator         : 0.000e+00
machine timescale operator  : 6.533e-17
vacuum flux                 : 1.580e-16
plasma flux                 : 0.000e+00
toroidal current density    : 3.626e-16
initial metal currents      : 0.000e+00
limiter-mask mismatches     : 0


In [5]:
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=70)
ax1.grid(True, which="both")
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()


### Prepare inputs for the PCS class

Here, we will set up the inputs (lists and dictionaries) for each of the (internal) controller classes within the virtual PCS (e.g. plasma, shape, virtual circuits). 

We start by defining which coils have which purpose.

In [6]:
# COIL NAMES AND SHAPE TARGET NAMES

# P6 is absent: all active circuits in this machine are even in Z.
active_coils = [
    "Solenoid", "PX", "D1", "D2", "D3", "Dp",
    "D5", "D6", "D7", "P4", "P5",
]
ctrl_coils = active_coils.copy()

# The PCS still evaluates its vertical controller, but its output is not mapped
# to a coil in this strictly symmetric example.
vertical_coils = []

plasma_targets = ["plasma"]

All of the following inputs are going to be specified as **waveforms**, i.e. they will be a dictionary of **times** and **vals**. 

For each array/list of **times** (in seconds), we must specify a corresponding list of list/arrays in **vals**. 

Most waveform quantities (e.g. `*_ref`, `*_ff`, `blend`) will use **linear** interpolation inside the PCS class internally, to enable querying at a given time $t$ within the simulation.

Some other quantities (e.g. PID gains, virtual circuits, and other arrays), will use **previous-value (or step)** interpolation where the value defined at time `tmin` is used over the interval $ [t_\mathrm{min},\, t_\mathrm{next})$, after which the value defined at `tnext` is applied.

We will denote which quantity uses which type of interpolation with **linear** or **step**. 

The following two cells provide an example of how to set these two different waveforms types and what they look like when plotted. 

In [7]:
# waveform dictionary structure (for linearly interpolated quantity)
# e.g. this could be the desired radial position of the X-point over time
waveform = {
    "times": np.array([0.0, 0.05, 0.15, 0.35, 0.45, 0.5]),
     "vals": np.array([0.57, 0.57, 0.5, 0.5, 0.57, 0.57]),
}

# plot what it looks like 
fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

ax.plot(waveform['times'], waveform['vals'], color='k', linewidth=1, linestyle="--", marker="x", markersize=7, label="linearly interpolated waveform")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Scalar quantity [units]")
ax.grid()
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/2355852286.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# waveform dictionary structure (for step interpolated quantity)
# e.g. this could be a proportional PID gain over time
waveform = {
    "times": np.array([0.0, 0.15, 0.3, 0.5]),
     "vals": np.array([100, 150, 75, 100]),
}

# plot what it looks like 
fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

ax.step(waveform['times'], waveform['vals'], where='post',
        color='k', linewidth=1, linestyle="--", marker="x", markersize=7,
        label="step interpolated waveform")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Scalar quantity [units]")
ax.grid()
ax.legend()
plt.tight_layout()
plt.show()


/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/424721672.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Before we define each of the controller settings, let us define some global times, i.e. the start and end of the simulation say. 

In [9]:
# choose some min amd max simulation times
tmin = 0.0
tmax = 0.25

First, we define the desired control settings for the **plasma category**.

These waveforms govern how the plasma current, $I_p$, is controlled—either via feedback (FB), feedforward (FF), or a combination of both.

The following quantities must be specified:

- **`ip_ref`**: Plasma current feedback reference waveform \[A\] (linear).
- **`vloop_ff`**: Loop voltage feedforward reference waveform \[V·s⁻¹\] (linear).
- **`blend`**: Control blending parameter [dimensionless] (linear):
  - `1` → purely FB control  
  - `0` → purely FF control  
  - values in `(0, 1)` → mixed FB/FF control.
- **`k_prop`**: Proportional gain waveform for the FB PID controller \[s⁻¹\] (step).
- **`k_int`**: Integral gain waveform for the FB PID controller \[s⁻²\] (step).
- **`k_deriv`**: Derivative gain waveform for the FB PID controller (no units) (step). 
- **`M_solenoid`**: Mutual inductance between the plasma and solenoid (required for FF control) \[V·s·A⁻¹\] (step).

In this particular example, we will tell the PCS to hold the plasma current constant at it's starting value. 

In [10]:
# PLASMA CATEGORY DATA

# build the dictionary which will be passed to the virtual PCS (all keys below are required)
plasma_data = {}

# plasma current feedback (FB) reference waveform 
# here we hold the value for the initial equilibrium constant
plasma_data["ip_ref"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([eq._profiles.Ip, eq._profiles.Ip])
    }

# loop voltage feedforward (FF) reference waveform (can specify the desired loop voltage on the plasma)
# (here we don't use a loop voltage as we will use the FB reference waveform above)
plasma_data["vloop_ff"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([0, 0])
    }

# blending waveform: tells controller to use purely FB control (1), purely FF control (0), or a mixture (between 0 and 1)
# we are doing pure FB control here
plasma_data["ip_blend"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([1, 1])
    }

# proportional gain for the FB PID controller
plasma_data["k_prop"] = {
    'times': np.array([tmin]),
    'vals': np.array([-5*4])
    }

# integral gain for the FB PID controller
plasma_data["k_int"] = {
    'times': np.array([tmin]),
    'vals': np.array([-50*2])
    }

# derivative gain for the FB PID controller
plasma_data["k_deriv"] = {
    'times': np.array([tmin]),
    'vals': np.array([0])
    }

# mutual inductance between plasma and solenoid: required for FF control
# not used here as we do not set a loop voltage
plasma_data["M_solenoid"] = {
    "times": np.array([0]), 
    "vals": np.array([1.0])
    }

plasma_data.keys()

dict_keys(['ip_ref', 'vloop_ff', 'ip_blend', 'k_prop', 'k_int', 'k_deriv', 'M_solenoid'])

Next, we define the desired control settings for the **shape category**.

First, we define a function that extracts the relevant shape parameters from the equilibrium object (here referred to as `plasma_descriptors`). In this example, we extract the following quantities:

- **`Rin`**: Inboard midplane radius.
- **`Rout`**: Outboard midplane radius.
- **`Rx`**: Radial position of the lower X-point.
- **`Zx`**: Vertical position of the lower X-point.

These are taken directly from the equilibrium object (without measurement noise). You could instead build synthetic diagnostics (e.g. magnetic probes) to take readings around the equilibrium (with measurement noise) and then reconstruct the plasma shape based on these. 

In [11]:
# define the descriptors function (it should return an array of values and take in an eq object)
# outputs ("measurements") from this function will be passed to the PCS later on during simulation
def plasma_descriptors(eq):

    # inboard/outboard midplane radii
    RinRout = eq.innerOuterSeparatrix()

    # find lower X-point
    # define a "box" in which to search for the lower X-point
    XPT_BOX = [[0.33, -0.88], [0.95, -1.38]]

    # mask those points
    xpt_mask = (
        (eq.xpt[:, 0] >= XPT_BOX[0][0])
        & (eq.xpt[:, 0] <= XPT_BOX[1][0])
        & (eq.xpt[:, 1] <= XPT_BOX[0][1])
        & (eq.xpt[:, 1] >= XPT_BOX[1][1])
    )
    xpts = eq.xpt[xpt_mask, 0:2].squeeze()
    if xpts.ndim > 1 and xpts.shape[0] > 1:
        opt = eq.opt[0, 0:2]
        dists = np.linalg.norm(xpts - opt, axis=1)
        idx = np.argmin(dists)  # index of closest point
        Rx, Zx = xpts[idx, :]
    else:
        Rx, Zx = xpts

    return np.array([RinRout[0], RinRout[1], Rx, Zx])

# give these descriptors some names for use in the PCS
ctrl_targets = ['Rin','Rout', 'Rx', 'Zx']

Next, we specify flat reference waveforms for the
shape parameters. Their values are taken directly from the initial equilibrium,
so this implementation test does not deliberately move the plasma or trigger
relinearisation through changing targets.

`Rin`, `Rout`, and `Rx` remain under feedback control. `Zx` retains a flat
reference for completeness but remains uncontrolled, as in example 11.

In [12]:
# SHAPE/DIVERTOR CATEGORY

shape_data = {}
initial_shape_values = dict(zip(ctrl_targets, plasma_descriptors(eq)))
shape_gains = {
    "Rin": 120,
    "Rout": 170,
    "Rx": 120,
    "Zx": 120,
}

for target in ctrl_targets:
    initial_value = initial_shape_values[target]
    shape_data[target] = {
        "ff": {
            "times": np.array([tmin, tmax]),
            "vals": np.array([0.0, 0.0]),
        },
        "ref": {
            "times": np.array([tmin, tmax]),
            "vals": np.array([initial_value, initial_value]),
        },
        "blend": {
            "times": np.array([tmin, tmax]),
            "vals": np.array([0.0, 0.0])
            if target == "Zx"
            else np.array([1.0, 1.0]),
        },
        "k_prop": {
            "times": np.array([tmin]),
            "vals": np.array([shape_gains[target]]),
        },
        "k_int": {
            "times": np.array([tmin]),
            "vals": np.array([0.0]),
        },
        "k_deriv": {
            "times": np.array([tmin]),
            "vals": np.array([0.0]),
        },
    }

shape_data.keys()

dict_keys(['Rin', 'Rout', 'Rx', 'Zx'])

Next, we define the schedule of virtual circuits (VCs) in the **virtual circuits category**.

For each shape parameter in the **shape category**, a VC is required to map the discrepancy between the target value (from the shape controller) and the measured value (from the equilibrium) to requests for the poloidal field (PF) coil currents. The VCs used here were obtained using the procedure described in the example notebook on VC construction. Recall that VCs are defined with units \[m/A\].

Each VC is an array with length equal to the number of `ctrl_coils`, with entries ordered consistently with that list. When specified in the waveform dictionary, VCs are **previous-value interpolated** by the PCS class. For example, for the `Rin` controller, the VC defined at time `tmin` is used over the interval $ [t_\mathrm{min},\, t_\mathrm{max})$, after which the VC defined at `tmax` is applied. Note here that we use only one VC for each shape parameter as the shape changes are not that significant. If more complex shape changes are required, then more VCs (linearised around the expected plasma shape), will be required. 

Note that the first entry of each shape-parameter VC is zero. This ensures that the solenoid does not contribute to shape control and is reserved exclusively for plasma current, $I_p$, control.

In addition to the shape VCs, a final VC is defined for the **plasma category**. This VC maps the requested change in $I_p$ from the plasma controller to PF coil current requests (well in this case, only a request to the solenoid).

Finally, optional pre-programmed feedforward (FF) requests for the PF coil currents may be specified. These allow the coil currents to be set directly if desired. In this example, no such FF requests are used, and the coil currents are driven purely via feedback control.


In [13]:
# VIRTUAL CIRCUITS CATEGORY

# build the dictionary which will be passed to the virtual PCS (all keys below are required)
circuits_data = {}

circuits_data["Rin"] = {
    "times": np.array([tmin]), 
    "vals": [
        np.array([0, 3.1394e+04,  8.5990e+03, -8.8300e+02, -1.2460e+03, -6.6690e+03, 1.2660e+03,  2.9770e+03,  3.5350e+03,  8.3110e+03, -7.8520e+03]),
        ]
    }

circuits_data["Rout"] = {
    "times": np.array([tmin]), 
    "vals": [
        np.array([0, -9.9700e+02,  1.1790e+03,  3.9700e+02, -2.0000e+02, -1.8970e+03, -5.5500e+02, -2.2640e+03, -1.5860e+03, -1.9750e+03,  5.5800e+03]),
        ]
    }

circuits_data["Rx"] = {
    "times": np.array([tmin]), 
    "vals": [
        np.array([0, -3.0021e+04,  3.3980e+03,  8.6900e+03,  5.8310e+03,  1.5858e+04, 2.4000e+01, -1.2100e+02, -2.2160e+03, -9.5290e+03,  1.3570e+03]),
        ]
    }

circuits_data["Zx"] = {
    "times": np.array([tmin]), 
    "vals": [
        np.array([0, 3.7670e+03,  2.0328e+04,  1.0562e+04,  4.2560e+03,  2.2600e+03, -2.0450e+03, -5.5680e+03, -5.3160e+03, -9.8440e+03,  3.3750e+03]),
        ]
    }

circuits_data["plasma"] = {
    "times": np.array([tmin]), 
    "vals": [
        np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        ]
    }

# store the coil order for future reference
circuits_data["coil_order"] = ctrl_coils

# define any feedforward coil current drives on each control coil (no drives used here)
zeros_dict = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
for coil in ctrl_coils: # linearly interpolated
    circuits_data[coil+"_ref"] = zeros_dict

circuits_data.keys()

dict_keys(['Rin', 'Rout', 'Rx', 'Zx', 'plasma', 'coil_order', 'Solenoid_ref', 'PX_ref', 'D1_ref', 'D2_ref', 'D3_ref', 'Dp_ref', 'D5_ref', 'D6_ref', 'D7_ref', 'P4_ref', 'P5_ref'])

Next, we define the inputs for the **systems category**.

This controller specifies operational limits for the coils listed in `ctrl_coils`, including:

- **`min_coil_curr_lims`**: Minimum allowable coil currents \[A\] (step).
- **`max_coil_curr_lims`**: Maximum allowable coil currents \[A\] (step).
- **`max_coil_curr_ramp_lims`**: Maximum allowable coil current ramp rates \[A·s⁻¹\] (step).

There is also the option to define feedforward (FF) perturbations (linear) to the control coil currents—similar to the FF drives available in the **virtual circuits category**. These are not used here.


In [14]:
# SYSTEMS CATEGORY

# build the dictionary which will be passed to the virtual PCS (all keys below are required)
systems_data = {}

# define the min coil current limits and ramp rate limits
systems_data["min_coil_curr_lims"] = {
    'times': [0.0],
    'vals': [np.array([-10000, -7000, -9000, -9000, -9000, -9000, -9000, -9000, -9000, -10000, -10000])],
    }

# define the max coil current limits
systems_data["max_coil_curr_lims"] = {
    'times': [0.0],
    'vals': [np.array([10000, 7000, 9000, 9000, 9000, 9000, 9000, 9000, 9000, 0, 0])],
    }

# define the max ramp rate limits for the coils
systems_data["max_coil_curr_ramp_lims"] = {
    'times': [0.0],
    'vals': [1e12*np.ones(len(ctrl_coils))],
    }

# define the control coil current perturbations
for name in ctrl_coils: # linearly interpolated
    systems_data[name+"_pert"] = zeros_dict

systems_data.keys()

dict_keys(['min_coil_curr_lims', 'max_coil_curr_lims', 'max_coil_curr_ramp_lims', 'Solenoid_pert', 'PX_pert', 'D1_pert', 'D2_pert', 'D3_pert', 'Dp_pert', 'D5_pert', 'D6_pert', 'D7_pert', 'P4_pert', 'P5_pert'])

Next, we define the inputs for the **PF category**.

This category specifies the electrical properties, gains, and operational limits of the poloidal field (PF) coils. These inputs are used by the virtual PCS to convert coil current requests into physically consistent voltage commands.

The following quantities must be provided:

- **`R_matrix`**: Coil resistance array for the PF coils \[Ω\] (step).  
  Defined as a time-dependent waveform and restricted to the coils listed in `ctrl_coils`.

- **`M_FF_matrix`**: Coil mutual inductance matrix used in feedforward (FF) terms \[Vs/A\] (step).  
  This accounts for inductive coupling between PF coils when computing FF voltage requests.

- **`M_FB_matrix`**: Coil mutual inductance matrix used in feedback (FB) terms \[Vs/A\] (step).
  This accounts for inductive coupling between PF coils when computing FF voltage requests. Often it is identical to `M_FF_matrix`, but is provided separately for flexibility.

- **`coil_gains`**: Feedback gain applied to each PF coil \[s\] (step).  
  These gains scale the feedback voltage contributions on a per-coil basis.

- **`coil_voltage_lims`**: Maximum allowable voltages for each PF coil \[V\] (step).  
  These limits are enforced by the PCS when generating coil voltage commands.

- **`coil_voltage_slew_lims`**: Maximum allowable voltage ramp rates for each PF coil \[V·s⁻¹\] (step).  
  In this example, very large values are used, effectively disabling slew-rate limiting.

All quantities are defined as time-dependent waveforms, even when constant, to allow future extension to time-varying PF system parameters.


In [15]:
# PF CATEGORY

# build the dictionary which will be passed to the virtual PCS (all keys below are required)
pf_data = {}

# coil resistances
pf_data["R_matrix"] = {
    'times': [0.0],
    'vals': [tokamak.coil_resist[0:11]],
    }

# coil mutual inductances (on feedforward terms)
pf_data["M_FF_matrix"] = {
    'times': [0.0],
    'vals': [tokamak.coil_self_ind[0:11,0:11]],
    }

# coil mutual inductances (on feedback terms)
pf_data["M_FB_matrix"] = {
    'times': [0.0],
    'vals': [tokamak.coil_self_ind[0:11,0:11]],
    }


# gains on the coils (for feedback term)
pf_data["coil_gains"] = {
    'times': [0.0],
    'vals': [0.015*np.ones(len(ctrl_coils))],
    }

# limits on voltages in the coils
pf_data["coil_voltage_lims"] = {
    "times": np.array([0.0]),
    'vals': [[2000,  750,  750,  750,  750,  750,  750,  750,  750,  750,  750]],
     } 

# limits on voltage ramp rates in the coils (we set no limits here)
pf_data["coil_voltage_slew_lims"] = {
    'times': [0.0],
    'vals': [1e12*np.ones(len(ctrl_coils))],
    }

pf_data.keys()

dict_keys(['R_matrix', 'M_FF_matrix', 'M_FB_matrix', 'coil_gains', 'coil_voltage_lims', 'coil_voltage_slew_lims'])

Next, we define the inputs for the **vertical category**.

The PCS interface still requires these data, but this strictly symmetric machine
has no vertical-control circuit. The controller output is therefore not mapped
to an active coil. The vertical reference remains zero and the evolutive solver
enforces symmetry directly.

In [16]:
# VERTICAL CATEGORY DATA

# build the dictionary which will be passed to the virtual PCS (all keys below are required)
vertical_data = {}

# Symmetry requires the vertical-position reference to remain at Z=0.
vertical_data["z_ref"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}

# proportional gain
vertical_data["k_prop"] = {"times": np.array([tmin]), "vals": np.array([0.025])}

# derivative gain
vertical_data["k_deriv"] = {"times": np.array([tmin]), "vals": np.array([5e-7])}

vertical_data.keys()

dict_keys(['z_ref', 'k_prop', 'k_deriv'])

Finally, we define the inputs for the **coil activations category**.

This category specifies when each retained, even-in-Z active circuit is enabled
or disabled. `P6` is absent from both the machine and this list.

In [17]:
# COIL ACTIVATIONS CATEGORY

# build the dictionary which will be passed to the virtual PCS (all keys below are required)
coil_activation_data = {}

# coil activation times
default_coil_dict = {"times": np.array([tmin]), "vals": np.array([1.0])}
for name in active_coils:
    coil_activation_data[name+"_activation"] = default_coil_dict

coil_activation_data.keys()

dict_keys(['Solenoid_activation', 'PX_activation', 'D1_activation', 'D2_activation', 'D3_activation', 'Dp_activation', 'D5_activation', 'D6_activation', 'D7_activation', 'P4_activation', 'P5_activation'])

### Initialise the virtual PCS class

Now that the inputs have been prepared we can initialise the class and use in-built functions to view or plot the waveform data in each control category. This will build internal classes for each of the above listed controllers. 

In [18]:
# FIRE UP THE PLASMA CONTROL SYSTEM
from freegsnke.control_loop.pcs import PlasmaControlSystem

PCS = PlasmaControlSystem(
    plasma_data=plasma_data,
    shape_data=shape_data,
    shape_control_mode="PID",
    circuits_data=circuits_data,
    systems_data=systems_data,
    pf_data=pf_data,
    vertical_data=vertical_data,
    coil_activation_data=coil_activation_data,
    active_coils=active_coils,
    ctrl_coils=ctrl_coils,
    vertical_coils=vertical_coils,
    ctrl_targets=ctrl_targets,
    plasma_target=plasma_targets,
)

In [19]:
PCS.CoilActivationController.data

{'Solenoid_activation': {'times': array([0.]), 'vals': array([1.])},
 'PX_activation': {'times': array([0.]), 'vals': array([1.])},
 'D1_activation': {'times': array([0.]), 'vals': array([1.])},
 'D2_activation': {'times': array([0.]), 'vals': array([1.])},
 'D3_activation': {'times': array([0.]), 'vals': array([1.])},
 'Dp_activation': {'times': array([0.]), 'vals': array([1.])},
 'D5_activation': {'times': array([0.]), 'vals': array([1.])},
 'D6_activation': {'times': array([0.]), 'vals': array([1.])},
 'D7_activation': {'times': array([0.]), 'vals': array([1.])},
 'P4_activation': {'times': array([0.]), 'vals': array([1.])},
 'P5_activation': {'times': array([0.]), 'vals': array([1.])}}

In [20]:
# each controller should contain a copy of the waveform dictionaries internally
PCS.PlasmaController.data

{'ip_ref': {'times': array([0.  , 0.25]), 'vals': array([759000., 759000.])},
 'vloop_ff': {'times': array([0.  , 0.25]), 'vals': array([0, 0])},
 'ip_blend': {'times': array([0.  , 0.25]), 'vals': array([1, 1])},
 'k_prop': {'times': array([0.]), 'vals': array([-20])},
 'k_int': {'times': array([0.]), 'vals': array([-100])},
 'k_deriv': {'times': array([0.]), 'vals': array([0])},
 'M_solenoid': {'times': array([0]), 'vals': array([1.])}}

In [21]:
# it will also contain a dictionary of functions that has linearly (or previous value) interpolated the various waveforms
PCS.PlasmaController.interpolants

{'ip_ref': <scipy.interpolate._fitpack2.InterpolatedUnivariateSpline at 0x32b67b0a0>,
 'vloop_ff': <scipy.interpolate._fitpack2.InterpolatedUnivariateSpline at 0x32b67ba60>,
 'ip_blend': <scipy.interpolate._fitpack2.InterpolatedUnivariateSpline at 0x32b67b970>,
 'k_prop': <scipy.interpolate._interpolate.interp1d at 0x32b596a70>,
 'k_int': <scipy.interpolate._interpolate.interp1d at 0x11787f5b0>,
 'k_deriv': <scipy.interpolate._interpolate.interp1d at 0x32b511ee0>,
 'M_solenoid': <scipy.interpolate._interpolate.interp1d at 0x3221d9b20>}

In [22]:
# # it should also be possible to plot the data (uncomment following cell)
# # green shading indiciates times that FB control is ON, yellow that FF is ON, a mix that both are ON, and white that there is no control
# PCS.PlasmaController.plot_data(tmin=tmin, tmax=tmax)

In [23]:
# the "run_control" method will be used internally later on but can also be called explicitly if required
PCS.PlasmaController.run_control

<bound method PlasmaController.run_control of <freegsnke.control_loop.plasma_category.PlasmaController object at 0x32b6d4520>>

Finally, suppose you wish to modify the shot setup data after initialising the PCS class. 

To do this, simply edit the waveforms required directly in the `.data` attribute of the controller of interest and then call `.update_interpolants()`. This has to be done to ensure that the controller refreshes any stale interpolants from a prior initialisation.

Uncomment the code below to see how it works for an example in the `ShapeController`.

In [24]:
# # update the data entry with your new waveform
# new_waveform = {"times": np.array([tmin, 0.05, 0.15, 0.35, 0.45, tmax]), "vals": np.array([0.28, 0.28, 0.30, 0.30, 0.28, 0.28])}  # we vary Rin
# PCS.ShapeController.data["Rin"]["ref"] = new_waveform

# # call the update function to refresh inteprolants (essential)
# PCS.ShapeController.update_interpolants()

# # plot to see new waveform
# PCS.ShapeController.plot_data(targ="Rin", tmin=tmin, tmax=tmax)

### Initialise the symmetry-reduced nonlinear solver

`force_up_down_symmetric=True` enforces an even plasma in every static solve.
The passive reflection map lets the solver identify and remove odd passive
normal modes before either the approximate or full plasma-coupling Jacobian is
built.

The timestep is five times that in example 11. The internal circuit timestep is
increased by the same factor; otherwise the implicit circuit solver would still
perform the old small substeps.

Example 11 retained 30 passive modes without distinguishing parity. This
example retains 15 even passive modes, giving a comparable passive-mode budget
after the odd half of the spectrum is removed. A relinearisation therefore
uses 27 current directions rather than 43; the four profile directions are
unchanged.

In [25]:
from freegsnke import nonlinear_solve

stepping = nonlinear_solve.nl_solver(
    eq=eq,
    profiles=profiles,
    GSStaticSolver=GSStaticSolver,
    full_timestep=2.5e-3,
    max_internal_timestep=2.5e-3,
    plasma_resistivity=1e-7,
    fix_n_vessel_modes=15,
    plasma_descriptor_function=plasma_descriptors,
    force_up_down_symmetric=True,
    passive_reflection_operator=passive_reflection_operator,
)

# Confirm that the retained basis and both plasma-response Jacobians are even.
retained_passive_parity = (
    stepping.evol_metal_curr.normal_modes.passive_mode_parity[
        stepping.evol_metal_curr.selected_modes_mask[n_active:]
    ]
)


def reduced_columns_reflection_error(columns):
    fields = np.zeros((stepping.nx, stepping.ny, columns.shape[1]))
    fields[stepping.limiter_handler.mask_inside_limiter] = columns
    return np.linalg.norm(fields - fields[:, ::-1]) / np.linalg.norm(fields)


jacobian_symmetry_errors = {
    "dIydI": reduced_columns_reflection_error(stepping.dIydI),
    "dIydtheta": reduced_columns_reflection_error(stepping.dIydtheta),
}
print("Retained passive mode parities:", np.unique(retained_passive_parity))
for name, error in jacobian_symmetry_errors.items():
    print(f"{name} reflection error: {error:.3e}")

assert np.all(retained_passive_parity == 1)
assert max(jacobian_symmetry_errors.values()) < 1e-14

-----
Checking that the provided 'eq' and 'profiles' are a GS solution...
Forward static solve SUCCESS. Tolerance 6.31e-09 (vs. requested 1.00e-08) reached in 0/100 iterations.
-----
Instantiating nonlinear solver objects...


done.
-----
Identifying mode selection criteria...
      'fix_n_vessel_modes' option selected --> passive structure modes that couple most to the strongest passive structure mode are being selected.
-----
Initial mode selection:
   Active coils
      total selected = 11 (out of 11)
   Passive structures
      15 selected using 'fix_n_vessel_modes'
   Total number of modes = 26 (11 active coils + 15 passive structures)
      (Note: some additional modes may be removed after Jacobian calculation if 'mode_removal=True')
-----


Building the 1510 x 27 Jacobian (dIy/dI) of plasma current density (inside the LCFS) with respect to all metal currents and the total plasma current.


Building the 1510 x 4 Jacobian (dIy/dtheta) of plasma current density (inside the LCFS) with respect to all plasma current density profile parameters within Jtor.


Built the 4 x 27 Jacobian (ds/dI) of plasma descriptors with respect to all metal currents and the total plasma current.
Built the 4 x 4 Jacobian (ds/dtheta) of plasma descriptors with respect to all plasma current density profile parameters within Jtor.
-----
Stability paramters:


      No unstable modes found in the retained even-in-Z subspace.
-----
Evolutive solver timestep:
      Solver timestep 'dt_step' has been set to 0.0025 as requested.
      Odd-in-Z modes are excluded; ensure the timestep resolves the retained even dynamics.
-----
Retained passive mode parities: [1]
dIydI reflection error: 0.000e+00
dIydtheta reflection error: 0.000e+00


### Define FPDT simulation parameters

Next set the key simulation parameters for the FPDT.

In [26]:
# FPDT SETUP

# number of simulation time steps
n = 100

# The odd vertical subspace has been removed, permitting a longer timestep.
dt = stepping.dt_step

# PCS time step (this must be whole fraction of the simulation time step)
# it governs the frequency at which the PCS is called
dt_PCS = dt/2

# # starting time (leave as it is)
# tmin = 0.0

# automatically calculates time array and sets time step in solver object
t_end = tmin + n*dt
times = np.arange(tmin, t_end, dt)

# (re-)initialise the dynamic solver with the initial eq and profiles
stepping.initialize_from_ICs(eq, profiles)

Before moving forward, we note that there are two time-dependent quantities that have not been set explicitly here. In this example, we hold the:

- plasma resistivity constant.
- plasma current density profiles constant.

For modelling real plasma discharges, both of these quantities should passed as time-dependent inputs to the time-stepping loop below. For further details, see the earlier example notebooks where time-dependent resistivity and profile evolution are demonstrated.


### FPDT Simulation

In the following cell, we initialise lists/arrays to store any equilbirium related data we wish to view after the simulation. See the FreeGSNKE example notebook (example03 - extracting_equilibrium_quantites) for a list of these and how to extract them. 

Note that some quantities can be computationally costly to extract and have not been optimised for speed yet!

In [27]:
# equilibrium-related data storage
dynamic_psi = np.zeros((stepping.eq1.psi().shape[0], stepping.eq1.psi().shape[1], len(times)))    # total poloidal flux
dynamic_limiter_flag = np.zeros(len(times))                                                       # flag if plasma is limited
dynamic_psi_boundary = np.zeros(len(times))                                                       # poloidal flux on plasma boundary
dynamic_xpts = []                                                                                 # list of X-point locations and associated poloidal flux
dynamic_opts = []                                                                                 # list of O-point locations and associated poloidal flux
dynamic_currents = np.zeros((len(stepping.vessel_currents_vec),len(times)))                         # PF coil and vessel eigenmode currents 
dynamic_ip = np.zeros(len(times))                                                                 # total plasma current
dynamic_shape_targets = np.zeros((len(ctrl_targets),len(times)))                                  # shape parameters
dynamic_triangularity = np.zeros(len(times))                                                      # plasma triangularity
linear_growth_rate = np.zeros(len(times))                                                         # linearised (deformable) growth rate estimate

# PCS-related data storage
V_approved = np.zeros((len(active_coils),len(times)))                                             # final PF coil voltages from the PCS class (passed into solver)
ip_hist = np.zeros(len(times))                                                                    # integral term feeding into Plasma Category PID FB controller (at prior time step)
ip_err = np.zeros(len(times))                                                                     # (error) for derivative term feeding into Plasma Category PID FB controller (at prior time step)
T_err = np.zeros((len(ctrl_targets),len(times)))                                                  # proportional term feeding into Shape Category PID FB controller (at prior time step)
T_hist = np.zeros((len(ctrl_targets),len(times)))                                                 # integral term feeding into Shape Category PID FB controller (at prior time step)
I_approved = np.zeros((len(ctrl_coils),len(times)))                                               # approved PF coil currents from prior time step feeding into System Category
coil_resists = np.zeros((len(active_coils),len(times)))                                           # PF coil resistances (to tell solver if a coil is switched on or off)
z_current = np.zeros(len(times))                                                                  # vertical position of the plasma (average jtor position)
dynamic_timings = np.zeros(len(times))                                                            # solver runtime at each time step
dynamic_jtor_norm = np.zeros(len(times))                                                          # norm change in jtor between time steps (used to trigger relinearisation)
threshold = 0.05                                                                                  # same relative-jtor relinearisation threshold used in example 11

# extract any initial values from the initial equilibrium
dynamic_psi[:,:,0] = stepping.eq1.psi()
dynamic_limiter_flag[0] = stepping.eq1._profiles.flag_limiter
dynamic_psi_boundary[0] = stepping.eq1._profiles.psi_bndry
dynamic_xpts.append(stepping.eq1.xpt)
dynamic_opts.append(stepping.eq1.opt)
dynamic_currents[:,0] = stepping.vessel_currents_vec.copy()
dynamic_ip[0] = stepping.profiles1.Ip
dynamic_shape_targets[:,0] = plasma_descriptors(eq=stepping.eq1)
dynamic_triangularity[0] = eq.triangularity()
linear_growth_rate[0] = np.max(np.real(stepping.linearised_sol.all_timescales))
z_current[0] = stepping.eq1.Zcurrent()

Before running the following cell (which takes a few minutes), be sure to familiarise yourself with all of the steps.

During each time step:

1. The primary method in the `PCS` class, `calculate_ctrl_voltages`, is called. Using “measurements” from the current equilibrium (plasma current, coil currents, and shape parameters), it computes the voltages to be applied to the evolutive solver, as well as the coil resistances (to determine whether any coils are switched off). These voltages enact the plasma control.

2. The plasma current density profile parameters are placed into a dictionary for use by the evolutive solver.  
   *Note:* they are constant here, but they can be made time-dependent at this stage.

3. The evolutive solver is then called using these parameters (along with additional ones). At this stage, several choices can be made:
   - Specify a time-dependent plasma resistivity (usually required for resimulation of prior discharges).
   - Select, via `linear_only`, either a linear or fully nonlinear solution of the circuit, plasma, and Grad–Shafranov equations.
   - Set a relinearisation threshold when using linear mode (see example notebook 05c).
   - Choose not to solve the Grad–Shafranov equation fully (when using linear mode), but instead solve only for the shape parameters specified in `plasma_descriptors` (see example notebooks 05b and 05c).

4. The required data are stored following successful completion of the time step.


In [28]:
# RUN FPDT SIMULATION
for i, t in enumerate(times[0:-1]):
    print("-----")
    print(f"t = {np.round(t,5)}s (step {i+1}/{n-1})")
    
    # start timer
    start_time = time.time()

    # initialise any historical quantities for PCS PID controllers
    if i == 0:
        ip_hist_prev = 0.0
        ip_err_prev = 0.0
        T_err_prev = np.zeros(len(ctrl_targets))
        T_hist_prev = np.zeros(len(ctrl_targets))
        I_approved_prev = eq.tokamak.getCurrentsVec()[0:11]  # must use starting currents here
        V_approved_prev = np.zeros(len(ctrl_coils))
        zipv_meas = 0.0
    else:
        ip_hist_prev=ip_hist[i-1].copy()                              # integral term from FB PID Plasma controller at prior step
        ip_err_prev=ip_err[i-1].copy()                                # error term from FB PID Plasma controller at prior step
        T_err_prev=T_err[:,i-1].copy()                                # proportional term from FB PID Shape controller at prior step
        T_hist_prev=T_hist[:,i-1].copy()                              # integral term from FB PID Shape controller at prior step
        I_approved_prev=I_approved[:,i-1].copy()                      # approved PF coil currents from prior step (ctrl_coils only)
        V_approved_prev=V_approved[:,i-1].copy()                   # approved PF coil voltages from prior step (ctrl_coils only)
        zipv_meas = ((z_current[i]-z_current[i-1])/dt)*dynamic_ip[i]  # rate of change of vertical plasma position 
    
    # call the PCS class to attain PF coil voltages (on ctrl_coils and vertical coil)
    V_approved[:,i], ip_hist[i], ip_err[i], T_err[:,i], T_hist[:,i], I_approved[:,i], coil_resists[:,i] = PCS.calculate_ctrl_voltages(
        t=t,                                        # current simulation time
        dt=dt_PCS,                                  # PCS time step
        dt_simulator=dt,                            # simulator (solver) time step
        ip_meas=dynamic_ip[i],                      # measured plasma current
        ip_hist_prev=ip_hist_prev,                  # plasma controller integral term history
        ip_err_prev=ip_err_prev,                    # plasma controller error term history (for derivative term)
        T_meas=dynamic_shape_targets[:,i].copy(),   # measured shape parameters
        T_err_prev=T_err_prev,                      # shape controller proportional term history
        T_hist_prev=T_hist_prev,                    # shape controller integral term history
        I_approved_prev=I_approved_prev,            # approved PF currents from prior timestep
        I_meas=dynamic_currents[0:11,i].copy(),     # measured PF coil currents
        V_approved_prev=V_approved_prev,            # approved PF voltages from prior timestep
        zip_meas=z_current[i]*dynamic_ip[i],        # measured vertical position x measured plasma current
        zipv_meas=zipv_meas,                        # derivative of above
        active_coil_resists=tokamak.coil_resist[0:11].copy(),    # PF coil resistances (these are constant)
        verbose=False,                              # print some output?
    )

    # extract plasma current density profile parameters (these are constant but can be time-dependent if desired)
    profile_params = {
            "alpha": profiles.alpha[0:2],
            "beta": profiles.beta[0:2],
            }

    # run freegsnke over the time step with the calculated voltages
    stepping.nlstepper(
        plasma_resistivity=1e-7,                          # assign plasma resistivity (chosen to be constant here)
        active_voltage_vec=V_approved[:,i],               # assign approved PF coil voltages from the PCS
        profiles_parameters=profile_params,               # assign profile parameters
        custom_active_coil_resistances=coil_resists[:,i], # assign active coil resistances from PCS (tells solver if any coils are switched off)
        linear_only=True,                                 # linear or nonlinear solve?
        target_relative_tol_currents=1e-2,                # relative tolerance in the currents required for convergence
        target_relative_tol_GS=1e-2,                      # relative tolerance in the plasma flux required for convergence
        working_relative_tol_GS=(1e-2)/2,                 # tolerance used when solving GS equation, expressed in terms of the change in the plasma flux due to one timestep of evolution (must be smaller tolerance above)
        max_solving_iterations=20,                        # stop after this many iterations
        verbose=False,                                    # print some output?
        relinearise_threshold=threshold,                  # if the relative jtor norm change from last linearisation is above threshold, relinearise around current equilibrium
        no_GS=False,                                      # do not solve GS at each time?
        )

    # stop timer
    end_time = time.time()

    # extract and store relevant data
    dynamic_psi[:,:,i+1] = stepping.eq1.psi()
    dynamic_limiter_flag[i+1] = stepping.eq1._profiles.flag_limiter
    dynamic_psi_boundary[i+1] = stepping.eq1._profiles.psi_bndry
    dynamic_xpts.append(stepping.eq1.xpt)
    dynamic_opts.append(stepping.eq1.opt)
    dynamic_currents[:,i+1] = stepping.vessel_currents_vec.copy()
    dynamic_ip[i+1] = stepping.currents_vec[-1] * stepping.plasma_norm_factor
    dynamic_shape_targets[:,i+1] = plasma_descriptors(stepping.eq1)
    dynamic_timings[i] = end_time - start_time
    dynamic_triangularity[i] = stepping.eq1.triangularity()
    z_current[i+1] = stepping.eq1.Zcurrent()
    dynamic_jtor_norm[i+1] = stepping.relinearise_criteria
    stepping.linearised_sol.calculate_linear_growth_rate()
    linear_growth_rate[i+1] = np.max(np.real(stepping.linearised_sol.all_timescales))

    # print some stuff to track solve
    print(f"      PLASMA QUANTITIES")
    print(f"        Ip = {np.round(dynamic_ip[i+1]/1000,1)} [kA]")
    print(f"        Shape parameters {ctrl_targets} = {np.round(dynamic_shape_targets[:,i+1],3)}")
    print(f"        Z position = {np.round(z_current[i+1],5)} [m]")
    print(f"        Linearised (deformable) growth rate = {np.round(1/linear_growth_rate[i+1],3)} [1/s]")
    print(f"      SOLVER STATUS")
    print(f"        Relinearisation status = {np.round(dynamic_jtor_norm[i+1],3)} (threshold = {threshold}) ")



# Verify that the complete evolution remained in the symmetric subspace.
flux_reflection_errors = [
    relative_reflection_error(dynamic_psi[:, :, i]) for i in range(len(times))
]
current_reflection_errors = []
for i in range(len(times)):
    current = dynamic_currents[:, i]
    denominator = np.linalg.norm(current)
    current_reflection_errors.append(
        0.0
        if denominator == 0
        else np.linalg.norm(reflection_operator @ current - current) / denominator
    )

print(
    "Maximum evolved flux reflection error:",
    f"{max(flux_reflection_errors):.3e}",
)
print(
    "Maximum evolved metal-current reflection error:",
    f"{max(current_reflection_errors):.3e}",
)
assert max(flux_reflection_errors) < 1e-12
assert max(current_reflection_errors) < 1e-12

-----
t = 0.0s (step 1/99)


      PLASMA QUANTITIES
        Ip = 757.9 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.277  1.346  0.567 -1.245]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.0 (threshold = 0.05) 
-----
t = 0.0025s (step 2/99)


      PLASMA QUANTITIES
        Ip = 757.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.277  1.346  0.568 -1.245]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.002 (threshold = 0.05) 
-----
t = 0.005s (step 3/99)
      PLASMA QUANTITIES
        Ip = 757.7 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.278  1.347  0.568 -1.245]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.002 (threshold = 0.05) 
-----
t = 0.0075s (step 4/99)
      PLASMA QUANTITIES
        Ip = 757.7 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.278  1.347  0.569 -1.245]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.004 (threshold = 0.05) 
-----
t = 0.01s (step 5/99)


      PLASMA QUANTITIES
        Ip = 757.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.245]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.006 (threshold = 0.05) 
-----
t = 0.0125s (step 6/99)
      PLASMA QUANTITIES
        Ip = 757.9 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.349  0.569 -1.246]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.009 (threshold = 0.05) 
-----
t = 0.015s (step 7/99)


      PLASMA QUANTITIES
        Ip = 758.0 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.28   1.349  0.57  -1.246]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.011 (threshold = 0.05) 
-----
t = 0.0175s (step 8/99)


...

      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.258]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.04 (threshold = 0.05) 
-----
t = 0.225s (step 91/99)
      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.258]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.041 (threshold = 0.05) 
-----
t = 0.2275s (step 92/99)


      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.258]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.041 (threshold = 0.05) 
-----
t = 0.23s (step 93/99)
      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.041 (threshold = 0.05) 
-----
t = 0.2325s (step 94/99)


      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.042 (threshold = 0.05) 
-----
t = 0.235s (step 95/99)
      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.042 (threshold = 0.05) 
-----
t = 0.2375s (step 96/99)


      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.042 (threshold = 0.05) 
-----
t = 0.24s (step 97/99)
      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.043 (threshold = 0.05) 
-----
t = 0.2425s (step 98/99)


      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.043 (threshold = 0.05) 
-----
t = 0.245s (step 99/99)
      PLASMA QUANTITIES
        Ip = 758.8 [kA]
        Shape parameters ['Rin', 'Rout', 'Rx', 'Zx'] = [ 0.279  1.348  0.569 -1.259]
        Z position = -0.0 [m]
        Linearised (deformable) growth rate = -8835.578 [1/s]
      SOLVER STATUS
        Relinearisation status = 0.043 (threshold = 0.05) 
Maximum evolved flux reflection error: 3.267e-16
Maximum evolved metal-current reflection error: 0.000e+00


### Plot some results

Let's now plot some of the output. We can compare the evolution of the controlled parameters (plasma current, shape parameters, and the constrained vertical position) against their FB reference waveforms. In these plots:
- the black dashed indicated the FB reference waveform. 
- the solid blue the simulated quantity.
- green background shading indicates when FB control is ON. 
- yellow background shading indicates when FF control is ON (not used). 
- white background shading indicates no control is used.

In [29]:
# PLASMA CURRENT EVOLUTION

fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# --- references and masks ---
FB_reference = PCS.PlasmaController.interpolants['ip_ref'](times)
FF_reference = PCS.PlasmaController.interpolants['vloop_ff'](times)

blend = PCS.PlasmaController.interpolants['ip_blend'](times)

FB_mask = (blend > 0) & (np.abs(FB_reference) > 0)
FF_mask = (blend < 1) & (np.abs(FF_reference) > 0)

# --- shade FB regions (green) ---
on_regions = np.where(np.diff(FB_mask.astype(int)) != 0)[0] + 1
for seg_t, seg_state in zip(np.split(times, on_regions),
                            np.split(FB_mask, on_regions)):
    if np.all(seg_state):
        ax.axvspan(seg_t[0], seg_t[-1],
                   color='green', alpha=0.25,
                   label="FB active")

# --- shade FF regions (yellow) ---
on_regions = np.where(np.diff(FF_mask.astype(int)) != 0)[0] + 1
for seg_t, seg_state in zip(np.split(times, on_regions),
                            np.split(FF_mask, on_regions)):
    if np.all(seg_state):
        ax.axvspan(seg_t[0], seg_t[-1],
                   color='gold', alpha=0.25,
                   label="FF active")

# --- FreeGSNKE ---
ax.plot(times, dynamic_ip,
        color='navy', linewidth=1,
        marker='x', markersize=0,
        label="FreeGSNKE")

# --- FB reference ---
ax.plot(times[FB_mask], FB_reference[FB_mask],
        color='k', linestyle='--',
        linewidth=1.5,
        label="FB reference")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Plasma current [A]")
ax.grid()

# deduplicate legend entries
handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(),
          dict(zip(labels, handles)).keys())

ax.set_ylim([7.54e5, 7.64e5])
fig.suptitle("Plasma Current", y=0.98)
plt.tight_layout()
plt.show()


/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/607465510.py:61: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [30]:
# PLASMA VERTICAL POSITION EVOLUTION

fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# --- FB reference ---
FB_reference = np.zeros_like(times)

# --- shade FB-active region (always on here) ---
ax.axvspan(times[0], times[-1],
           color='green', alpha=0.2,
           label="FB active")

# --- references ---
ax.plot(times, FB_reference,
        color='k', linestyle='--',
        linewidth=1.5,
        label="FB reference")

# --- FreeGSNKE ---
ax.plot(times, z_current,
        color='navy', linewidth=1,
        marker='x', markersize=0,
        label="FreeGSNKE")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Vertical position [m]")
ax.grid()
ax.legend()

fig.suptitle("Vertical Position", y=0.98)
plt.tight_layout()
plt.show()


/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/3578980179.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [31]:
# SHAPE PARAMETER EVOLUTION
# All shape references remain fixed at their initial values in this test.

ntarg = len(ctrl_targets)

fig, axes = plt.subplots(
    nrows=ntarg,
    ncols=1,
    figsize=(8, 12),
    dpi=80,
    sharex=True
)

for j, targ in enumerate(ctrl_targets):
    ax = axes[j]

    # --- references and masks ---
    FF_reference = PCS.ShapeController.interpolants[targ]['ff'](times)
    FB_reference = PCS.ShapeController.interpolants[targ]['ref'](times)

    FF_mask = (
        (PCS.ShapeController.interpolants[targ]['blend'](times) < 1)
        & (np.abs(PCS.ShapeController.interpolants[targ]['ff'].derivative()(times)) > 0)
    )

    FB_mask = (
        (PCS.ShapeController.interpolants[targ]['blend'](times) > 0)
        & (np.abs(FB_reference) > 0)
    )

    # --- shade FB regions (green) ---
    on_regions = np.where(np.diff(FB_mask.astype(int)) != 0)[0] + 1
    for seg_t, seg_state in zip(np.split(times, on_regions),
                                np.split(FB_mask, on_regions)):
        if np.all(seg_state):
            ax.axvspan(seg_t[0], seg_t[-1], color='green', alpha=0.25,
                       label="FB active" if j == 0 else None)

    # --- shade FF regions (yellow) ---
    on_regions = np.where(np.diff(FF_mask.astype(int)) != 0)[0] + 1
    for seg_t, seg_state in zip(np.split(times, on_regions),
                                np.split(FF_mask, on_regions)):
        if np.all(seg_state):
            ax.axvspan(seg_t[0], seg_t[-1], color='gold', alpha=0.25,
                       label="FF active" if j == 0 else None)

    # --- references ---
    ax.plot(times[FB_mask], FB_reference[FB_mask],
            color='k', linestyle='--', linewidth=1.5,
            label="FB reference" if j == 0 else None)

    if np.any(FF_mask):
        ax.plot(times[FF_mask], FF_reference[FF_mask],
                color='r', linestyle='--', linewidth=1.5,
                label="FF reference" if j == 0 else None)

    # --- FreeGSNKE ---
    ax.plot(times, dynamic_shape_targets[j, :],
            color='navy', linewidth=1,
            marker='x', markersize=0,
            label="FreeGSNKE" if j == 0 else None)

    ax.set_ylabel(f"{ctrl_targets[j]} [m]")
    ax.grid()
    ax.set_ylim([np.min(FB_reference)-0.02, np.max(FB_reference)+0.02])

# shared x label
axes[-1].set_xlabel(r"Shot time [$s$]")

# legend once
axes[0].legend(ncol=4, loc="upper right")

fig.suptitle("Shape parameters", y=0.995, fontsize=14)
plt.tight_layout()
plt.show()

/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/3687543762.py:75: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# PLASMA TRIANGULARITY EVOLUTION 

fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# --- FreeGSNKE ---
ax.plot(times[0:-1], dynamic_triangularity[0:-1],
        color='navy', linewidth=1,
        marker='x', markersize=0,
        label="FreeGSNKE")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Triangularity")
ax.grid()
ax.legend()

plt.tight_layout()
plt.show()


/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/3380605802.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


We can also show the voltages produced by the PCS and the subsequent evolution of the currents for each of the PF coils. To display the voltage/current limits, uncomment the relevant sections in the next cell.

In [33]:
# PF COIL VOLTAGES AND CURRENTS

ncoils = len(active_coils)

fig, axes = plt.subplots(
    nrows=ncoils,
    ncols=2,
    figsize=(16, 45),
    dpi=80,
    sharex=True
)

for j, coil in enumerate(active_coils):
    axV = axes[j, 0]  # voltage column
    axI = axes[j, 1]  # current column

    # VOLTAGES (left)

    # # voltage limits
    # vlim = PCS.PFController.data['coil_voltage_lims']['vals'][0][j]
    # axV.hlines([-vlim, vlim], times[0], times[-1],
    #            colors='k', linestyles='--', linewidth=1.2,
    #            label="Coil limits" if j == 0 else None)

    axV.plot(times[0:-1], V_approved[j, 0:-1],
             color='navy', linewidth=1,
             marker='x', markersize=0,
             label="FreeGSNKE")

    axV.set_ylabel(f'{coil} voltage [V]')
    axV.grid()
    if j == 0:
        axV.legend()

    # CURRENTS (right)

    # # current limits
    # imin = PCS.SystemsController.data['min_coil_curr_lims']['vals'][0][j]
    # imax = PCS.SystemsController.data['max_coil_curr_lims']['vals'][0][j]
    # axI.hlines([imin, imax], times[0], times[-1],
    #            colors='k', linestyles='--', linewidth=1.2,
    #            label="Coil limits" if j == 0 else None)

    axI.plot(times, dynamic_currents[j, :],
             color='navy', linewidth=1,
             marker='x', markersize=0,
             label="FreeGSNKE")

    axI.set_ylabel(f'{coil} current [A]')
    axI.grid()
    if j == 0:
        axI.legend()

# x-labels only on bottom row
axes[-1, 0].set_xlabel(r'Shot time [$s$]')
axes[-1, 1].set_xlabel(r'Shot time [$s$]')

# column titles
axes[0, 0].set_title("PF Coil Voltages")
axes[0, 1].set_title("PF Coil Currents")

plt.tight_layout()
plt.show()


/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/4159241437.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Given that we have passive conducting structures in the machine, we can also plot the evolution of their currents. Note how they all start from zero given that we did not initialise them in this example. 

In [34]:
# PASSIVE CURRENTS EVOLUTION

# enter the names of some passives here
passives_to_plot = ["vessel_1", "centrecolumn_1", "gas_baffle_upper_1", "p5_case_upper_1"]
 
# indices of passives_to_plot in tokamak.coil_names
indxs = [tokamak.coil_names.index(name) for name in passives_to_plot]
 
fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)
 
# --- FreeGSNKE ---
for name, idx in zip(passives_to_plot, indxs):
    ax.plot(
        times, dynamic_currents[idx, :],
        linewidth=1,
        marker='x', markersize=0,
        label=name,
    )
 
ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Currents [A]")
ax.grid()
ax.legend()
 
fig.suptitle("Passive currents evolution", y=0.98)
plt.tight_layout()
plt.show()
 

/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/1642535016.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The odd vertical dynamics have been removed from
this model, so a vertical-instability timescale is no longer the criterion used
to choose the evolution timestep. The following diagnostic instead shows the
remaining linearised timescales.

In [35]:
# RETAINED LINEARISED TIMESCALES

fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# plot results
ax.plot(times[0:len(linear_growth_rate)], linear_growth_rate, color='navy', linewidth=1, marker=None, markersize=3, label="FPDT")
ax.hlines(dt, xmin=times[0], xmax=times[-1], color='black', linestyle='--', linewidth=1.5, label=f"dt")


ax.set_ylim([1e-4, 1e-1])
ax.set_yscale('log')
# ax.legend()
ax.grid()
ax.set_xlabel(r'Shot time [$s$]')
ax.set_ylabel('Linearised instability timescale [s]')
fig.suptitle("Retained linearised timescales", y=0.98)
plt.tight_layout()
plt.show()



/var/folders/_y/dm0w4q1j76scm_xhwnz9bsz80000gp/T/ipykernel_61848/460088166.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The following cell can be uncommented to make an animation of the equilibrium in the machine over time. Might take a few seconds to run and will save the output .mp4 in the `data` directory. 

In [36]:
# # SAVE AN ANIMATION OF THE SIMULATION
# %matplotlib inline

# import matplotlib.animation as animation

# plt.rcParams['figure.dpi'] = 90

# fig1, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# # Plot static wall / tokamak background
# eq.tokamak.plot(axis=ax1, show=False)
# ax1.plot(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2)
# ax1.grid(True, which='both', alpha=0.5)
# ax1.set_aspect('equal')
# ax1.set_xlabel(r'Major radius, $R$ $[m]$')
# ax1.set_ylabel(r'Height, $Z$ $[m]$')
# ax1.set_xlim(0.05, 2.15)
# ax1.set_ylim(-2.25, 2.25)

# # Determine contour levels from entire dynamic_psi
# min_psi = np.min(dynamic_psi)
# max_psi = np.max(dynamic_psi)
# levels = np.linspace(min_psi, max_psi, 40)

# # --- Storage for dynamic artists ---
# contour_artists = []
# scatter_artists = []

# # --- Update function ---
# def update(i):
    
#     global contour_artists, scatter_artists

#     # Remove previous dynamic artists
#     for c in contour_artists + scatter_artists:
#         if isinstance(c, list):
#             for coll in c:
#                 coll.remove()
#         else:
#             c.remove()
#     contour_artists = []
#     scatter_artists = []

#     ax1.set_title(rf"$t$ = {np.round(times[i],3)}")

#     # Main psi contours
#     c1 = ax1.contour(eq.R, eq.Z, dynamic_psi[:,:,i], levels=levels, alpha=0.8, cmap='viridis')
#     contour_artists.append(c1)

#     # plasma boundary
#     c2 = ax1.contour(eq.R, eq.Z, dynamic_psi[:,:,i],
#                          levels=[dynamic_psi_boundary[i]],
#                          linestyles="-", colors='r', linewidths=1.4)
#     contour_artists.append(c2)

#     # adds separatrix of primary X-point if plasma limited
#     if dynamic_limiter_flag[i]:
#         c3 = ax1.contour(eq.R, eq.Z, dynamic_psi[:,:,i],
#                             levels=[dynamic_xpts[i][0,2]],
#                             linestyles="--", colors='k', linewidths=1.4)
#         contour_artists.append(c3)


#     # X-points and O-points
#     sc1 = ax1.scatter(dynamic_xpts[i][:,0], dynamic_xpts[i][:,1], color='r', marker='x', s=30)
#     sc2 = ax1.scatter(dynamic_opts[i][:,0], dynamic_opts[i][:,1], color='g', marker='2', s=30)

#     # indicate which is the primary X-point more clearly
#     sc3 = ax1.scatter(dynamic_xpts[i][0,0], dynamic_xpts[i][0,1], color='r', marker='x', s=60)

#     scatter_artists.extend([sc1, sc2, sc3])

#     return contour_artists + scatter_artists

# # --- Animation ---
# num_frames = len(times)
# frames_to_plot = range(0, num_frames, 5)
# fps = int(len(frames_to_plot)/10)
# ani = animation.FuncAnimation(fig1, update, frames=frames_to_plot, interval=1, blit=True)

# # save to video (much faster & smaller than GIF)
# ani.save(f"data/animated_equilibrium.mp4", writer="ffmpeg", fps=fps)
